In [ ]:
import sys; sys.path.append('..')
import sheet_convergence, sim_utils
import MeshFEM
from tri_mesh_viewer import TriMeshViewer
import numpy as np

import meshing, mesh, elastic_solid, energy

pts, _ = sheet_convergence.stripBoundary(1)
m = mesh.Mesh(*meshing.tetrahedralize_extruded_polylines([np.array(pts + [pts[0]])], [], thickness=4.0, maxVol=0.001), degree=1)
es = elastic_solid.ElasticSolid(m, energy.NeoHookeanYoungPoisson(3, 1, 0))

x_orig = es.getVars().copy()
x = x_orig.copy()

es_re = elastic_solid.ElasticSolidRotExtrap(m, energy.NeoHookeanYoungPoisson(3, 1, 0))

In [ ]:
v = TriMeshViewer(es, wireframe=True)
v.setCameraParams(((3.6996509275435927, 1.3705005944233914, 4.1946387036007655),
 (-0.18400815367026574, 0.9706515767088857, -0.15484352106373026),
 (0.0, 0.0, 0.0)))
v.show()

In [ ]:
import compute_vibrational_modes
#lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(es, n=12, mtype=compute_vibrational_modes.MassMatrixType.IDENTITY, sigma=-1e-11)
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(es, n=12, mtype=compute_vibrational_modes.MassMatrixType.FULL, sigma=-1e-10)

In [ ]:
es_re.setVars(x_orig)
es_re.energy()

In [ ]:
es_re.method = es_re.method.ElementExtrapolation
# es_re.method = es_re.method.ModalWarping

In [ ]:
v2 = TriMeshViewer(es_re, wireframe=True)
v2.show()

In [ ]:
v2.setCameraParams(v.getCameraParams())

In [ ]:
es_re.setVars(x_orig)
v2.update()

In [ ]:
es_re.setVars(x_orig + 0.3 * modes[:, 10])
v2.update()

In [ ]:
es_re.method = es_re.method.ElementExtrapolation
v2.update()

In [ ]:
es_re.method = es_re.method.ModalWarping
v2.update()

In [ ]:
es_re.setVars(x_orig)

In [ ]:
v = modes[:, 0]
eps = 1e-1
es_re.elasticSolid.setVars(x_orig + eps * v)
es_re.setVars(x_orig + eps * v)
x_extrap_plus = es_re.elasticSolid.getVars().copy()
es_re.elasticSolid.setVars(x_orig - eps * v)
es_re.setVars(x_orig - eps * v)
x_extrap_minus = es_re.elasticSolid.getVars().copy()
x_extrap_velocity = (x_extrap_plus - x_extrap_minus) / (2 * eps)

In [ ]:
v2.update()

In [ ]:
import mode_viewer

In [ ]:
mview = mode_viewer.ModeViewer(es, modes, lambdas, wireframe=False, numSteps=2)

In [ ]:
mview.show()

In [ ]:
mview.amplitude = 0.5

In [ ]:
mview_esre = mode_viewer.ModeViewer(es_re, modes, lambdas, wireframe=False)

In [ ]:
mview_esre.amplitude = 0.5

In [ ]:
es_re.method = es_re.method.ElementExtrapolation

In [ ]:
es_re.method = es_re.method.ModalWarping

In [ ]:
mview_esre.show()

In [ ]:
from PIL import ImageDraw
from PIL import ImageFont

In [ ]:
from PIL import ImageChops

In [ ]:
# def renderModeImages(v, modes, amplitude, width=None, height=None, normalCreaseAngle = np.pi / 4):
width, height = 2048, 2048
zFightOffset=5e-4
amplitude = 0.5
kwargs = {}
images = []
normalCreaseAngle = np.pi / 4
if width is not None: kwargs['width'] = width
if height is not None: kwargs['height'] = height
orender = v.offscreenRenderer(**kwargs)
zFightOffsetVec = zFightOffset * np.linalg.inv(orender.matView @ orender.meshes[0].matModel)[0:3, 2]
currVars = es.getVars()
vg_undefo = es.visualizationGeometry(normalCreaseAngle)
for mode in modes.T:
    es.setVars(currVars + amplitude * mode)
    vg_defo = es.visualizationGeometry(np.pi / 4)
    V =  (vg_defo[0] + zFightOffsetVec)[vg_defo[1].ravel()]
    N =  vg_defo[2]
    C = np.repeat([[0.6, 0.8, 1.0, 0.5]], 3 * len(vg_defo[1]),   axis=0)
    F = np.arange(len(V), dtype=np.uint32)
    orender.addMesh(V, F, N, C, makeDefault=False)
    orender.meshes[-1].matModel = orender.meshes[0].matModel.copy()
    orender.meshes[0].alpha = 1.0
    orender.meshes[0].lineWidth = 0.0
    orender.render()
    orender.removeMesh(-1)
    images.append(orender.image().resize((768, 768)))
# # Common crop
# bbox = 
# for i, l in enumerate(lambdas):
#     img = images[i]
#     draw = ImageDraw.Draw(img)
#     #draw.text((img.width / 2, 20), f"Mode {i} - λ = {l}", fill=(0, 0, 0), font=ImageFont.truetype('fonts/FreeSans.ttf', size=20), anchor="mm")
#     print(img.getbbox())

In [ ]:
img.getbbox?

In [ ]:
images[11]

In [ ]:
images[0].tell

In [ ]:
images[10]